**[Source]** New (통합 프로젝트 신규) — 오류 분석(12번)에서 확인된 정답 라벨 문제 후보를 규칙으로 점검
**[Status]** NEW
**[Role]** 정답 라벨 감사(label audit). 확실한 구어체 축약형 몇 개에 대해 ‘입력의 축약형이 정답에도 그대로 남은 행’을 후보로 표시. 원본 라벨은 수정하지 않고 제안·근거·검수 필요 여부만 data/curated/에 저장
**[Modification]** 지수 FIXED 데이터를 읽기만 한다. 결과 파일은 별도 폴더(data/curated)에 쓰며 Train/Validation만 다룬다(Test 라벨은 열지 않는다). 이 감사 결과로 재학습하지 않았으므로 최종 모델과 무관하다.
**규칙 기반 후보 표시일 뿐이며 사람이 검수하기 전에는 정답 라벨로 채택하지 않는다.**

# 12b. 정답 라벨 감사 (규칙 기반, 선택 단계)
- 목적: 모델 오류 중 일부가 정답 라벨 문제일 수 있으므로(12번), **높은 정밀도의 규칙**으로만 후보를 찾는다.
- 규칙(어절 전체가 일치할 때만): `걍→그냥`, `그니까→그러니까`, `글고→그리고`, `어케→어떻게`, `왤케→왜 이렇게`.
- **절대 자동으로 바꾸지 않는 것**: `ㅋㅋ`, `ㅎㅎ`, `ㅠㅠ`, `헐`, `대박` 같은 감탄·이모티콘성 표현, 숫자, 영문, 이름 토큰.
- 각 규칙이 얼마나 믿을 만한지는 **다른 행의 실제 정답이 그 규칙을 따르는 비율**로 확인한다(정답이 이미 교정한 행 vs 축약형을 남긴 행).
- 출력: `data/curated/label_audit_{train,validation}.csv` — 입력, 원래 정답, 규칙 제안, 최종 라벨(=원래 정답, 미변경), 라벨 출처, 변경 이유, 안전 플래그, 검수 필요 여부.

In [1]:
# [공통 준비] 경로 · 재현성 · 05번 검증 통과 확인
import os, sys, json, re, time, math, random, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70); pd.set_option("display.unicode.east_asian_width", True)

def _find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "config" / "paths.json").exists():
            return c
    raise FileNotFoundError("config/paths.json이 있는 통합 프로젝트 루트를 찾지 못했습니다(notebooks 폴더에서 실행하세요).")
ROOT = _find_root(); sys.path.insert(0, str(ROOT / "src"))
import common, ko_metrics as km
P = common.load_paths(ROOT)
SEED = 42; random.seed(SEED); np.random.seed(SEED)

VER = json.loads((P.PROCESSED / "verification_05" / "verification_result.json").read_text(encoding="utf-8"))
assert VER["verdict"] == "PASS", "05번 전처리 검증이 PASS가 아닙니다 → 모델링을 진행하지 않습니다"
MAN = common.manifest(P)
assert VER["sha256_actual"]["train"] == MAN["sha256"]["train.jsonl"], "검증 이후 데이터가 바뀌었습니다(05번을 다시 실행하세요)"
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "|", platform.platform())
print("통합 프로젝트:", ROOT); print("지수 최종 데이터:", P.DATA_DIR, "|", MAN["dataset_version"])
print("05번 검증:", VER["verdict"], "@", VER["verified_at"], "| metric backends:", km.BACKENDS)

Python 3.10.12 | pandas 2.3.3 | numpy 2.2.6 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
통합 프로젝트: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction
지수 최종 데이터: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/data/preprocessed_final | preprocessed_final_v1
05번 검증: PASS @ 2026-09-21 05:30:57 | metric backends: {'rapidfuzz': False, 'sacrebleu': False}


In [2]:
# [셀 1] 규칙 정의 + Train/Validation 적용
import re
RULES = {"R1_걍": ("걍", "그냥"), "R2_그니까": ("그니까", "그러니까"), "R3_글고": ("글고", "그리고"), "R4_어케": ("어케", "어떻게"), "R5_왤케": ("왤케", "왜 이렇게")}
TOK = re.compile(r"[^\s]+")
def has_token(text, tok):        # 어절 전체 일치(뒤에 붙은 문장부호는 허용)
    return any(re.fullmatch(re.escape(tok) + r"[.,!?~]*", w) for w in text.split())
def safety(t):
    f = []
    if re.search(r"[ㅋㅎㅠㅜ]", t): f.append("자음·감탄 표현 포함")
    if re.search(r"[0-9]", t): f.append("숫자 포함")
    if re.search(r"[A-Za-z]", t): f.append("영문 포함")
    if re.search(r"[\U0001F300-\U0001FAFF]", t): f.append("이모지 포함")
    return ";".join(f)
def audit(split):
    R = common.read_split(P, split, columns=["document_id", "utterance_id", "input", "target"]); out = []; stat = {k: {"입력에 포함": 0, "정답이 교정함": 0, "정답이 축약형 유지": 0} for k in RULES}
    for r in R:
        for k, (a, b) in RULES.items():
            if not has_token(r["input"], a): continue
            stat[k]["입력에 포함"] += 1
            if has_token(r["target"], a):
                stat[k]["정답이 축약형 유지"] += 1
                sug = " ".join(re.sub("^" + re.escape(a), b, w) if re.fullmatch(re.escape(a) + r"[.,!?~]*", w) else w for w in r["target"].split())
                out.append({"utterance_id": r["utterance_id"], "document_id": r["document_id"], "split": split, "input": r["input"], "original_target": r["target"], "rule_id": k, "rule_suggestion": sug,
                            "final_label": r["target"], "label_source": "original(미변경)", "change_reason": f"입력의 '{a}'가 정답에 그대로 남음 → '{b}' 교정 누락 후보", "safety_flags": safety(r["target"]), "review_required": True})
            elif b in r["target"]:
                stat[k]["정답이 교정함"] += 1
    return pd.DataFrame(out), pd.DataFrame(stat).T, len(R)
res = {}
for sp in ("train", "validation"):
    df, st, n = audit(sp); res[sp] = (df, st, n); print(f"[{sp}] 행 수 {n:,} | 감사 후보 {len(df):,}행"); st["정답이 교정한 비율"] = (st["정답이 교정함"] / st["입력에 포함"]).round(3); print(st.to_string()); print()

[train] 행 수 986,718 | 감사 후보 4,777행
           입력에 포함  정답이 교정함  정답이 축약형 유지  정답이 교정한 비율
R1_걍             2491            240                2243               0.096
R2_그니까         1708              4                1695               0.002
R3_글고            457            168                 288               0.368
R4_어케            482             93                 368               0.193
R5_왤케            228             42                 183               0.184

[validation] 행 수 54,730 | 감사 후보 326행
           입력에 포함  정답이 교정함  정답이 축약형 유지  정답이 교정한 비율
R1_걍              181             19                 161               0.105
R2_그니까          107              0                 106               0.000
R3_글고             31             13                  18               0.419
R4_어케             28              2                  25               0.071
R5_왤케             18              1                  16               0.056



In [3]:
# [셀 2] 저장 (원본 데이터는 수정하지 않는다) + 사례
P.CURATED.mkdir(parents=True, exist_ok=True)
for sp, (df, st, n) in res.items():
    df.to_csv(P.CURATED / f"label_audit_{sp}.csv", index=False, encoding="utf-8-sig"); print("저장:", P.CURATED / f"label_audit_{sp}.csv", len(df), "행")
tr = res["train"][0]; print("\n[Train 후보 사례 — 규칙별 최대 3건]")
for k in RULES:
    for _, r in tr[tr.rule_id == k].head(3).iterrows(): print(f" {k} | 입력: {r.input} | 정답: {r.original_target} | 제안: {r.rule_suggestion}")
sm = {sp: {"rows": n, "candidates": len(df), "rule_stats": st.reset_index().rename(columns={"index": "rule"}).to_dict("records")} for sp, (df, st, n) in res.items()}
(P.RUNS / "label_audit_12b.json").write_text(json.dumps(sm, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print("\n※ 이 감사 결과로 학습·평가 데이터를 바꾸지 않았다. Test 라벨은 열지 않았다:", [f for f in common.OPENED_FILES])

저장: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction/data/curated/label_audit_train.csv 4777 행
저장: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction/data/curated/label_audit_validation.csv 326 행

[Train 후보 사례 — 규칙별 최대 3건]
 R1_걍 | 입력: 이래서 걍 타고 감 | 정답: 이래서 걍 타고 감. | 제안: 이래서 그냥 타고 감.
 R1_걍 | 입력: 난 걍 그거 먹고 개맛없다 이러고 말텐데 | 정답: 난 걍 그거 먹고 개맛없다 이러고 말 텐데. | 제안: 난 그냥 그거 먹고 개맛없다 이러고 말 텐데.
 R1_걍 | 입력: 걍 처박힘 떡볶이 | 정답: 걍 처박힘, 떡볶이. | 제안: 그냥 처박힘, 떡볶이.
 R2_그니까 | 입력: 그니까 | 정답: 그니까. | 제안: 그러니까.
 R2_그니까 | 입력: 그니까~~~~~ | 정답: 그니까~~~~~. | 제안: 그러니까~~~~~.
 R2_그니까 | 입력: 그니까 | 정답: 그니까. | 제안: 그러니까.
 R3_글고 | 입력: 글고 딱 타이밍 맞게 | 정답: 글고 딱 타이밍 맞게, | 제안: 그리고 딱 타이밍 맞게,
 R3_글고 | 입력: 글고 주택하나가지고 이중 대출 안댜 ㅜㅜㅜ | 정답: 글고 주택 하나 가지고 이중 대출 안 돼. ㅜㅜㅜ | 제안: 그리고 주택 하나 가지고 이중 대출 안 돼. ㅜㅜㅜ
 R3_글고 | 입력: 글고 난 키워드알림이 안되서ㅠㅠ | 정답: 글고 난 키워드 알림이 안 돼서. ㅠㅠ | 제안: 그리고 난 키워드 알림이 안 돼서. ㅠㅠ
 R4_어케 | 입력: 가운데 사람 앞머리 어케 만들어? | 정답: 가운데 사람 앞머리 어케 만들어? | 제안: 가운데 사람 앞머리 어떻게 만들어?
 R4_어케 | 입력: 어케

## 해석
- **규칙 가정이 이 데이터의 라벨 관례와 맞지 않는다(가설 기각).** 규칙은 “구어체 축약형은 정답에서 표준어로 바뀌어야 한다”는 가정에 서 있었다. 그러나 Train에서 입력에 `걍`이 있는 2,491행 중 정답이 `그냥`으로 바꾼 행은 240행(9.6%), `그니까`는 1,708행 중 4행(0.2%), `어케`는 19.3%, `왤케` 18.4%, `글고` 36.8%뿐이고 나머지는 축약형을 **그대로 유지**했다. Validation도 같은 패턴(걍 10.5%, 그니까 0%)이다. 이 말뭉치의 정답은 띄어쓰기·문장부호·오탈자는 고치지만 구어체 어휘 자체는 대체로 보존하는 것으로 보인다(라벨 관례에 대한 추정이며, 이번에는 가이드라인 문서를 확인하지 않았다).
- 따라서 이 규칙으로 정답을 `그냥` 등으로 바꾸면 **오히려 데이터의 일관성을 깨뜨린다**. 후보 파일(Train 4,777행, Validation 326행)은 “규칙 위반 후보”가 아니라 “규칙 가정을 검증하기 위한 대조 자료”로만 남긴다. `review_required=True`, `final_label=원래 정답(미변경)`으로 저장했고, **어떤 라벨도 수정하지 않았다**.
- 12번 오류 분석의 ‘미교정’ 사례(`걍 밑반찬이랑 …` 정답이 `그냥`으로 교정됨)는 소수(약 10%)의 라벨이 다수 관례와 어긋난 경우일 가능성이 있다. 즉 같은 입력이 라벨마다 다르게 교정된 **라벨 불일치**가 미교정·부분교정 오류의 일부일 수 있다(빈도는 측정하지 않았다).
- 결론: 라벨 감사 규칙 기반 정제 → **채택하지 않음**. 라벨 정제를 통한 재학습은 수행하지 않았고 최종 모델과 무관하다.